# Customer Intelligence & Predictive Retention Platform
## Notebook 01 — Exploratory Data Analysis (EDA)

### Project Objective
This notebook explores customer churn patterns in the IBM Telco Customer Churn dataset. The goal is to understand customer characteristics, identify churn-related patterns, and derive actionable business insights.

### Dataset
- **Source:** IBM Telco Customer Churn dataset
- **Records:** 7,043 customers
- **Features:** 21 columns
- **Target variable:** `Churn`

### Analysis Performed
1. Dataset structure and data quality
2. Customer churn distribution
3. Contract type and churn rate
4. Customer tenure analysis
5. Monthly charges analysis
6. Business insights and retention recommendations

### Tools and Libraries
Python, Pandas, NumPy, Matplotlib, Seaborn, and Jupyter Notebook.

### Important Note
The relationships identified in this notebook are observational. They describe patterns in this dataset and do not establish that any single feature causes customer churn.


In [ ]:

# Customer Intelligence & Predictive Retention Platform
# Notebook 01 — Exploratory Data Analysis (single-cell version)

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --------------------------------------------------
# 1. Paths and setup
# --------------------------------------------------
cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "reports").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Project root not found. Run this notebook from the project "
        "root or its notebooks folder."
    )

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Telco-Customer-Churn.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully!")
print("Project root:", PROJECT_ROOT)

# --------------------------------------------------
# 2. Load dataset
# --------------------------------------------------
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print("\nDATASET LOADED")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())

# --------------------------------------------------
# 3. Data quality overview
# --------------------------------------------------
print("\nDATASET INFORMATION")
print("=" * 50)
print("Shape:", df.shape)
df.info()

print("\nMissing values:")
display(
    df.isnull().sum()
      .sort_values(ascending=False)
      .to_frame("Missing Values")
)

print("\nDuplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

if "Churn" not in df.columns:
    raise ValueError("Required column 'Churn' not found.")

# --------------------------------------------------
# 4. Churn distribution
# --------------------------------------------------
churn_summary = (
    df["Churn"].value_counts()
    .rename_axis("Churn")
    .reset_index(name="Customer_Count")
)
churn_summary["Percentage"] = (
    churn_summary["Customer_Count"] / len(df) * 100
).round(2)

print("\nCUSTOMER CHURN DISTRIBUTION")
display(churn_summary)

fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(data=df, x="Churn", ax=ax)
ax.set_title("Customer Churn Distribution")
ax.set_xlabel("Churn")
ax.set_ylabel("Number of Customers")
fig.tight_layout()
fig.savefig(
    REPORTS_DIR / "notebook_churn_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

churn_summary.to_csv(
    REPORTS_DIR / "notebook_churn_summary.csv", index=False
)

# --------------------------------------------------
# 5. Contract type and churn
# --------------------------------------------------
if "Contract" not in df.columns:
    raise ValueError("Required column 'Contract' not found.")

df["ChurnFlag"] = (df["Churn"] == "Yes").astype(int)

contract_summary = (
    df.groupby("Contract")
      .agg(
          Total_Customers=("ChurnFlag", "size"),
          Churned_Customers=("ChurnFlag", "sum"),
          Churn_Rate=("ChurnFlag", "mean")
      )
      .reset_index()
)
contract_summary["Churn_Rate"] = (
    contract_summary["Churn_Rate"] * 100
).round(2)

print("\nCHURN BY CONTRACT")
display(contract_summary)

contract_summary.to_csv(
    REPORTS_DIR / "notebook_contract_churn.csv", index=False
)

contract_rates = contract_summary.set_index("Contract")["Churn_Rate"]
contract_rates = contract_rates.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=contract_rates.index, y=contract_rates.values, ax=ax)
ax.set_title("Customer Churn Rate by Contract Type")
ax.set_xlabel("Contract Type")
ax.set_ylabel("Churn Rate (%)")
ax.bar_label(ax.containers[0], fmt="%.1f%%")
fig.tight_layout()
fig.savefig(
    REPORTS_DIR / "notebook_contract_churn.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# --------------------------------------------------
# 6. Tenure analysis
# --------------------------------------------------
if "tenure" not in df.columns:
    raise ValueError("Required column 'tenure' not found.")

tenure_summary = (
    df.groupby("Churn")["tenure"]
      .agg(["count", "mean", "median"])
      .round(2)
)

print("\nTENURE BY CHURN")
display(tenure_summary)

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(
    data=df, x="tenure", hue="Churn",
    bins=30, multiple="stack", ax=ax
)
ax.set_title("Customer Tenure Distribution by Churn")
ax.set_xlabel("Tenure (months)")
ax.set_ylabel("Number of Customers")
fig.tight_layout()
fig.savefig(
    REPORTS_DIR / "notebook_tenure_churn.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# --------------------------------------------------
# 7. Monthly charges analysis
# --------------------------------------------------
if "MonthlyCharges" not in df.columns:
    raise ValueError("Required column 'MonthlyCharges' not found.")

charges_summary = (
    df.groupby("Churn")["MonthlyCharges"]
      .agg(["count", "mean", "median"])
      .round(2)
)

print("\nMONTHLY CHARGES BY CHURN")
display(charges_summary)

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=ax)
ax.set_title("Monthly Charges by Customer Churn Status")
ax.set_xlabel("Churn")
ax.set_ylabel("Monthly Charges ($)")
fig.tight_layout()
fig.savefig(
    REPORTS_DIR / "notebook_monthly_charges_churn.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# --------------------------------------------------
# 8. Business insights
# --------------------------------------------------
overall_churn_rate = (df["Churn"] == "Yes").mean() * 100
avg_tenure = df.groupby("Churn")["tenure"].mean()
avg_charges = df.groupby("Churn")["MonthlyCharges"].mean()

print("\nBUSINESS INSIGHTS")
print("=" * 50)
print(f"Total customers: {len(df):,}")
print(f"Overall churn rate: {overall_churn_rate:.2f}%")

if "Yes" in avg_tenure.index and "No" in avg_tenure.index:
    print(f"Average tenure - Churned: {avg_tenure['Yes']:.2f} months")
    print(f"Average tenure - Retained: {avg_tenure['No']:.2f} months")

if "Yes" in avg_charges.index and "No" in avg_charges.index:
    print(f"Average monthly charges - Churned: ${avg_charges['Yes']:.2f}")
    print(f"Average monthly charges - Retained: ${avg_charges['No']:.2f}")

print("\nRecommended business actions:")
print("1. Review retention offers for month-to-month customers.")
print("2. Investigate early-tenure customer experience.")
print("3. Review pricing and service concerns among high-charge customers.")
print("4. Validate findings before launching retention campaigns.")

# --------------------------------------------------
# 9. Save customer summary and final output list
# --------------------------------------------------
customer_summary = (
    df.groupby("Churn")
      .agg(
          Customer_Count=("Churn", "size"),
          Average_Tenure=("tenure", "mean"),
          Average_Monthly_Charges=("MonthlyCharges", "mean")
      )
      .round(2)
      .reset_index()
)

customer_summary.to_csv(
    REPORTS_DIR / "notebook_customer_summary.csv", index=False
)

print("\nEDA outputs saved successfully!")
print("Report location:", REPORTS_DIR)

for file in sorted(REPORTS_DIR.glob("notebook_*")):
    print("-", file.name)
